## IMPACT DE LA MISE EN SERVICE D'UNE NOUVELLE STATION DE METRO SUR LE PRIX DE L'IMMOBILIER
### Projet 2A - Python pour la data science 
*Sonny Augusto, Abel Cornet-Carlos et Mayténa Labinsky*

## INTRODUCTION
Rennes est l'une des plus petites villes au monde à disposer d'un réseau de métro automatique. Parmi les nombreux critères qui déterminent le prix d'un bien immobilier, l'accessibilité aux transports en commun est un facteur différenciant. Dans ce contexte, ce projet analyse dans quelle mesure la mise en service d'une nouvelle ligne de métro impacte la valeur de l'immobilier des quartiers desservis. Notre objectif est de quantifier précisément la plus-value immobilière (en pourcentage) générée par la proximité immédiate d'une nouvelle station.

## Sommaire
- [Installation](#installation)
- [Préparation des Données](#préparation-des-données)
- [Statistiques Descriptives](#statistiques-descriptives)
- [Modèle de prédiction de la plus-value](#modèle-de-prédiction-du-prix)
- [Conclusion](#conclusion)

## Installation

In [ ]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
#Fonctions:
import src.clear_data as cd
import src.get_data as gd
import src.analyse_data as ad
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify

# Préparation des données

Adresses

In [ ]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf2021 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2021/full.csv.gz"

url_dvf2022 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2022/full.csv.gz"

url_dvf2023 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

url_dvf2024 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/full.csv.gz"

url_dvf2025 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2025/full.csv.gz"

Import et modifications

In [ ]:
#Import
df_dvf_raw2021 = gd.fetch_dvf_api(url_dvf2021)
df_dvf_raw2022 = gd.fetch_dvf_api(url_dvf2022)
df_dvf_raw2023 = gd.fetch_dvf_api(url_dvf2023)
df_dvf_raw2024 = gd.fetch_dvf_api(url_dvf2024)
df_dvf_raw2025 = gd.fetch_dvf_api(url_dvf2025)
gdf_metro_raw = gd.fetch_metro_api(url_metro)


Analyse des bases de données en vu de les nettoyer

Analyse des 5 dvfs en vue de la fusion:
On regarde si elles contiennent bien les mêmes colonnes



In [ ]:
list_dvf = [
    df_dvf_raw2021, 
    df_dvf_raw2022, 
    df_dvf_raw2023, 
    df_dvf_raw2024, 
    df_dvf_raw2025
]
labels = ["2021", "2022", "2023", "2024", "2025"]
ad.verify_dvf_columns(list_dvf, labels)


Une fois qu'on a vu que les colonnes étaient identiques, on les mets bout à bout.

In [ ]:
#fusion
df_dvf_raw = cd.merge_yearly_dvf(list_dvf)
df_dvf_raw.to_csv("data/merged_raw.csv", index=False)

Nettoyage de la base


Pour faciliter l'analyse des données dans la suite, on nettoie les deux bases de données: on enlève les lignes contenant des valeurs manquantes, on ne garde dans la base dvf que les colonnes utiles pour l'analyse, et que les ventes d'appartements et de maisons, on rajoute aussi le prix au m² pour faciliter les analyses.

In [ ]:
#Nettoyage (projection, gestion des valeurs manquantes et sélection des colonnes utiles et rajout du prix/m²)
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
gdf_dvf['prix_m2'] = gdf_dvf['valeur_fonciere'] / gdf_dvf['surface_reelle_bati']

Analyse et suppression des valeurs extrêmes

Pour ne pas que les analyses soient faussées par des vente de 0 m² ou des prix trop élevés, on enlève de la base de donnée les valeurs extrêmes du prix au m² et de la superficie.

In [ ]:
seuils_surface=ad.extreme_value_surface(gdf_dvf)

In [ ]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "surface_reelle_bati", seuils_surface[0], seuils_surface[1])

In [ ]:
seuils_prix=ad.extreme_value_prix(gdf_dvf)

In [ ]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "prix_m2", seuils_prix[0], seuils_prix[1])

On fusionne maintenant la base pour obtenir une seule base, avec à chaque ligne la distance à la station de métro la plus proche pour la ligne a et pour la ligne b.

In [ ]:
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)

## Statistiques Descriptives 

In [ ]:
print(gdf_final.columns)

In [ ]:
print(gdf_final.describe())

In [ ]:
import src.stats_desc as sd

# Statistiques générales
print("--- Statistiques Globales ---")
print(sd.get_general_stats(gdf_final))

# Comparaison Ligne A vs Ligne B
print("\n--- Comparaison par Ligne de Métro ---")
stats_lignes = sd.get_stats_by_ligne(gdf_final)
print(stats_lignes)

# Statistique par tranche (distance min au métro <250, 250-500, 500-800, >800)
print("\n--- Statistique par tranche ---")
stats_tranches = sd.analyse_prix_dist_tranche(gdf_final)
print(stats_tranches)


#Graphique baton des statistiques par tranche
sd.plot_prix_par_tranche(stats_tranches)

# Statistique par variable de contrôle
print("\n--- Statistique par variable de contrôle ---")
var_control_type = sd.compare_proximity_controlled(gdf_final)
print(var_control_type)


In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=gdf_final, x="longitude", y="latitude", hue="prix_m2", 
                palette="YlOrRd", size="prix_m2", sizes=(10, 100), alpha=0.6)
plt.title("Répartition géographique des prix au m² à Rennes")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify # Nécessaire pour les ruptures de Jenks

# 1. Calculer les classes de Jenks (ex: 5 classes)
classifier = mapclassify.NaturalBreaks(gdf_final['prix_m2'], k=5)

# 2. Créer une nouvelle colonne de catégories basée sur ces classes
# On utilise les intervalles comme étiquettes pour la légende
gdf_final['prix_jenks'] = gdf_final['prix_m2'].apply(classifier.find_bin)

# 3. Création du graphique
plt.figure(figsize=(12, 8))

# Utilisation de la nouvelle colonne 'prix_jenks' pour la couleur (hue)
plot = sns.scatterplot(
    data=gdf_final, 
    x="longitude", 
    y="latitude", 
    hue="prix_jenks", 
    palette="YlOrRd", 
    size="prix_m2", 
    sizes=(20, 200), 
    alpha=0.7,
    edgecolor="black",
    linewidth=0.5
)

# 4. Personnalisation de la légende pour afficher les intervalles réels
labels = [f"Classe {i}: {interval}" for i, interval in enumerate(classifier.get_legend_classes())]
plt.legend(title="Prix au m² (Jenks)", labels=labels, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title("Répartition géographique des prix au m² à Rennes (Classification de Jenks)")
plt.tight_layout()
plt.show()

In [ ]:
sns.regplot(data=gdf_final[gdf_final['prix_m2'] < 10000], 
            x="dist_min_metro", y="prix_m2", 
            scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.title("Relation entre prix au m² et distance au métro")
plt.xlabel("Distance au métro le plus proche (m)")
plt.show()

DiD

In [ ]:
df_ready = sd.prepare_did_data(gdf_final)
results = sd.run_did_regression(df_ready)
print(results.summary())

In [ ]:
df_ready[['treated', 'post_event']].value_counts()

In [ ]:
# plot_did_trends(df_ready)

## Modèle de prédiction du prix 

In [ ]:
from src.model import preparer_et_entrainer

# Supposons que df_rennes et gdf_metro sont déjà chargés
model_final, features_utilisees = preparer_et_entrainer(gdf_final, gdf_metro)

print("Modèle entraîné avec succès !")
print(f"Features utilisées : {features_utilisees}")

# Vous pouvez maintenant utiliser model_final pour faire des prédictions
# exemple_data = ... 
# prediction = model_final.predict(exemple_data)

In [ ]:

from src.model import preparer_et_entrainer, predire_impact_nouvelle_station

# Exemple : un appartement de 50m², 3 pièces, en 2026
surface = 50
pieces = 3
annee = 2026
code_type = 1 # Supposons que 1 soit l'appartement

# Scénario 1 : Le métro est à 2 km (2000m)
prix_actuel = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=2000)

# Scénario 2 : Le métro est à 0 m (nouvelle station en bas de l'immeuble)
prix_avec_metro = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=0)

print(f"Prix estimé actuel : {prix_actuel:.2f} €")
print(f"Prix estimé avec nouvelle station : {prix_avec_metro:.2f} €")
print(f"Plus-value théorique : {prix_avec_metro - prix_actuel:.2f} €")